<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/Chapter_09B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_09B:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Chapter 9 Analysis of Variance (ANOVA)**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* 9.1 : Introduction
* 9.2 : Analysis of Variance (ANOVA)
* **9.3 : The Assumptions of ANOVA**
* **9.4 : Advanced**

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 KEY

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA1403 key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 key: {e}")
    print("Please set your STA1403 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 Key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 28
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your STA1403 key is correctly installed.

## Load Data Sets

Run the next cell to load the data sets for this lesson.

In [ ]:
# @title Load Data Sets
import pandas as pd

# Load dataset
cushings_df = pd.read_csv("https://biologicslab.co/STA1403/data/Cushings.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
genotype_df = pd.read_csv("https://biologicslab.co/STA1403/data/genotype.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
anorexia_df = pd.read_csv("https://biologicslab.co/STA1403/data/anorexia.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
cabbages_df = pd.read_csv("https://biologicslab.co/STA1403/data/cabbages.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

print("Data sets loaded sucessfully. ✅")

# **Chapter 9 : Analysis of Variance (ANOVA)**

## **9.3 The Assumptions of ANOVA**

To use ANOVA models, we assume that the samples are selected randomly from the population and independently from each other (e.g., by using simple random sampling). Further, we assume that the response variable in each group has a normal distribution. While the means of these normal distributions can change from one group to another, we assume that they all have the same variance. This is the same assumption we used for pooled $t$-tests.

Violation of these assumptions could lead to wrong inference. The independence assumption is violated, for example, if we obtain multiple observations (e.g., over a period of time) for each subject in our sample. In this case, we need to use **repeated-measures ANOVA** (not discussed in this book), which can be regarded as a generalization of paired $t$-tests. The consequence of violating the normality assumption is not very severe as long as the sample sizes for all the groups are large enough so the distributions of the sample means are approximately normal due to the central limit theorem. Finally, the assumption of equal variance among all groups is usually unrealistic and is often violated in practice. This is clearly the case for the example discussed in this chapter (Fig. 9.2). In practice, you are likely to see problems more similar to this example (maybe not as severe) that problems where the assumption of equal variance holds. Similar to the normality assumption, the results of ANOVA are not severely affected if the group variances moderately differ from each other. Alternatively, we could use ANOVA models without the equal variance assumption. See Weerahandi (2003) [38] for example.

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09B_image01A.png)

>**Fig. 9.8** Plot of means for the log of Tetrahydrocortisone by syndrome type

Sometimes, we can stabilize the variance (i.e., making it approximately constant) by using simple data transformations such as log or square root. For the example discussed above, using the log-transformation of `Tetrahydrocortisone` (instead of the original variable) makes the equal variance assumption more reasonable (Fig. 9.8). In Example 1 below, we use Python to create a new variable by taking the log of `Tetrahydrocortisone` and then perform analysis of variance for this newly created variable.

## Example 1: Plot Log of Means

In [ ]:
# @title Example 1: Plot Log of Means

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define parameter
anova_factor = "TCort"
factor_name = "Tetrahydrocortisone"

# Create log variable
log_df=cushings_df.copy()
log_df['logFactor'] = np.log(log_df[anova_factor])

# 1. Calculate mean and 95% CI (approx 1.96 * SEM)
# Group by Type and calculate mean and standard error of the mean (SEM)
stats = log_df.groupby('Type')['logFactor'].agg(['mean', 'sem']).reset_index()
stats['ci'] = stats['sem'] * 1.96 * 2

# Ensure the order of Types is correct (a, b, c, u)
stats['Type'] = pd.Categorical(stats['Type'], categories=['a', 'b', 'c', 'u'], ordered=True)
stats = stats.sort_values('Type')

# 2. Plotting
plt.figure(figsize=(6, 5), facecolor='white')
plt.rcParams['axes.facecolor'] = 'white'

# Plot means as a line graph with black circle markers
plt.plot(stats['Type'], stats['mean'], color='black', marker='o', markersize=8, linestyle='solid', linewidth=1.0)

# 3. Add vertical dashed error bars
plt.errorbar(stats['Type'], stats['mean'], yerr=stats['ci'], fmt='none',
             ecolor='black', capsize=4, linestyle='dotted', linewidth=0.4)

# 4. Labels and Title
plt.title('Plot of Means', fontsize=14)
plt.xlabel('Type', fontsize=12)
plt.ylabel(f'Log({factor_name})', fontsize=12)

# 5. Minimalist style
plt.grid(False)
plt.gca().spines['top'].set_visible(True)
plt.gca().spines['right'].set_visible(True)
plt.gca().tick_params(axis='both', colors='black')

plt.tight_layout()


If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09B_image02A.png)

In this case, the observed value of $F$-statistic is $f = 7.6$, and the corresponding $p$-value is $0.001$.

## **Exercise 1: Plot Log of Means**

In the cell below, plot the log of means for the metabolite `Pregnanetriol` by syndrome type.

**Code Hints:**

1. Copy-and-paste Example 1 in the cell below.
2. Change the variable `anova_factor` to "PregN"
3. Change the variable `factor_name` to "Pregnanetriol"

In [ ]:
# @title Exercise 1: Plot Log of Means



If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09B_image03A.png)

## **9.4 Advanced**

In this section, we briefly discuss ANOVA with two factors. We also provide some useful Python functions to perform ANOVA.

## **_9.4.1 Two-Way ANOVA_**

Consider the study by Bailey (1953) to investigate the inheritance of maternal influences on the growth of the rat [29]. In this study, rat litters were separated from their natural mothers, and they were nurtured by foster mothers. Mothers and litters can have four different genotypes: A, B, I, and J.

Suppose that we want to investigate whether weight gain (`Wt`) of the litter (in grams) at age 28 days is related to foster mother's genotype (`Mother`). (Rat litters were separated from their natural mothers at birth and given to foster mothers.) For this, we could use the one-way ANOVA procedure to compare the means of weight gain across different groups (genotypes). That is, we regard `Wt` as the response variable and `Mother` as the factor. For this example, however, we might want to take into account the genotype of rat litters (`Litter`) as well. The litter's genotype is itself a factor, and even though it is not the main factor in this study, it should be included in the analysis since we believe that it could influence the relationship between the main factor, mother's genotype, and the response variable, weight gain.

An ANOVA with two factors is called a **two-way ANOVA**. (In general, we can have a multi-way ANOVA by including multiple factors.) In many two-way ANOVA procedures, one of the two factors is the main explanatory variable of interest. The other factor is included since it is believed to be important in the study of the relationship between the main factor and the response variable. This is the case for the "rat genotype" example. In this example, we are mainly interested in the variation of weight gain across different genotypes of mothers. However, we need to account for possible weight gain variation due to the genotype of the litters. By including both factors `Mother` and `Litter`, we are dividing the total variation, $SS$, into three sources: (1) variation explained by the mother's genotype, $SS_{M}$, (2) variation explained by the litter's genotype, $SS_{L}$, and (3) the random variation, $S_{E}$, of weight gain not explained by either mother's genotype or litter's genotype. (Note that we have switched our notation from $SS_{W}$ to $SS_{E}$.) So,

$$
SS = SS_{M} + SS_{L} + SS_{E}.
$$

This type of two-way ANOVA is commonly used for experiments with a randomized block design discussed in Sect. 1.7. For these experiments, the treatment variable is the factor whose effect on the response variable is of main interest. The categorical variable used for blocking is the factor which is believed to be important, but its relationship with the response variable is not the focus of the experiment.

In the next subsection, we show how two-way ANOVA models can be implemented in Python. Here, we discuss how Python can be used for two-way ANOVA problems, where both factors are considered to be important, and we are interested in learning how the relationship between one factor with the response variable changes depending on the value of the other factor. For example, we might be interested in the effect of three different diets, $A$, $B$, and $C$, on blood pressure, but we believe that the diet effect varies between male and female groups. In this case, we say that there is an **interaction** between the two factors diet and gender.

For the “rat genotype” example, suppose we believe that the relationship between mother’s genotypes and weight gain changes depending on litter’s genotype. Therefore, we need to consider possible interaction between `Mother` and `Litter`. We use $M\times L$ to denote this interaction. Including the interaction between the two factors in a two-way ANOVA means that we believe a part of the total variation SS is explained by the combination of the two factors. That is, we can write the total variation as follows:

$$
SS = SS_{M} + SS_{L} + SS_{M \times L} + SS_{E}.
$$

The variation in the response variable due to specific combinations of the two factors is usually referred to as the **interaction effect**. In contrast, the variation in the response variable due to one of the factors alone (i.e., regardless of the values of the other factor) is called the **main effect**.

Now we use Python to perform two-way ANOVA for the “rat genotype” as shown in the following Demonstration.

## Demonstration: Two-Way ANOVA with Interation

The code in the cell below performs a two-way ANOVA on the `genotype` dataset to test whether there is a possible interaction between `Mother` and `Litter` on the dependent variable `Wt`. In this demonstration, we use the function `ols()` with formula = `Wt ~ C(Mother) * C(Litter)`. The asterisk `*` tell the `ols()` function to test for the **interaction** between `Mother` and `Litter`. If we don't want to test for an interaction, we use the plus symbol `+` instead of an `asterisk`.


In [ ]:
# @title Demonstration: Two-Way ANOVA with Interaction

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Define your variables
dependent_var = "Wt"
factor_1 = "Mother"
factor_2 = "Litter"
dataset = genotype_df
interaction_symbol = "*"  # asterisk for interaction


# 1. Define the R-style significance function
def get_sig_stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    elif p_value < 0.1:
        return "."
    else:
        return " "

# 2, Define the formula
formula = f"{dependent_var} ~ C({factor_1}) {interaction_symbol} C({factor_2})"
print(f"ols() formula = {formula}\n")

# 3. Run Two-Way ANOVA
model = ols(formula, data=dataset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# 4. Add the significance stars column based on 'PR(>F)'
anova_table['Signif'] = anova_table['PR(>F)'].apply(get_sig_stars)

# 4. Print the updated table
print(f"ANOVA Table (Type II tests)")
print(f"\nResponse: {dependent_var}")
print(anova_table)
print("\nSignif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")

If the code is correct, you should see the following output:
```text
ols() formula = Wt ~ C(Mother) * C(Litter)

ANOVA Table (Type II tests)

Response: Wt
                          sum_sq    df         F    PR(>F) Signif
C(Mother)             775.080588   3.0  4.763246  0.005736     **
C(Litter)              63.632488   3.0  0.391052  0.760004       
C(Mother):C(Litter)   824.072512   9.0  1.688108  0.120053       
Residual             2440.816500  45.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

The interpretation of sum of squares, degrees of freedom, $F$ -statistic, and $p$-value is similar to one-way ANOVA. In this example, $SS_M = 775.08$, $SS_L = 63.63$, and $SS_{M×L} = 824.07$. Based on these results, only the relationship between mother’s genotype and weigh gain is statistically significant at 0.05 level (pobs = 0.006). The interaction effect (shown as `C(Litter):C(Mother)` and the main effect of litter’s genotype are not statistically significant at 0.05 level.

## **_9.4.2 ANOVA Using Python_**

After we obtain $f$, the observed value of the $F$-statistic, we can use the $F$-distribution to obtain the corresponding $p$-value by calculating the upper tail probability of $f$. The functions $df()$, $pf()$, and $qf()$ provide the density, tail probability, and quantiles from the $F$-distribution with given degrees of freedom. For the Cushing's examples, $f = 3.2$, and the degrees of freedom are $df_{1} = 3$ and $df_{2} = 23$ with an upper tail probability of $3.2$.

Alternatively, we can use the Python function `ols()` to perform ANOVA directly. For this, we specify the response and factor variables using the same formula notation we used for the $t$-test: `response ~ factor`. Here, the response variable is `Tetrahydrocortisone`, and the factor is `Type` as demonstrated in Example 1 below.


## _One-Way ANOVA_

## Example 1: One-Way ANOVA

The code in the cell below shows how to performa one-way ANOVA using the `Cushings` dataset. In this example, we are testing whether the secretion of the metabolite Pregnanetriol (`PregN`) is dependent upon the tumor type (`Type`). The formula for the `ols()` function is: `ols() formula = PregN ~ Type`.

In [ ]:
# @title Example 1: One-Way ANOVA

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Select variables
anova_factor_1 = "PregN"
factor_1_name = "Pregnanetriol"
anova_factor_2 = "Type"
dataset = cushings_df

# 1. Define the R-style significance function
def get_sig_stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    elif p_value < 0.1:
        return "."
    else:
        return " "
# 2. Combine variables into a single string variable
formula = f"{anova_factor_1} ~ {anova_factor_2}"
print(f"ols() formula = {formula}\n")

# 3. Perform ANOVA
model = ols(formula, data=dataset).fit()

# 4. Generate ANOVA table
anova_table = sm.stats.anova_lm(model, typ=2)

# 3. Add the significance stars column based on 'PR(>F)'
anova_table['Signif'] = anova_table['PR(>F)'].apply(get_sig_stars)

# 4. Print the updated table
print(anova_table)
print("\nSignif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")

If the code is correct, you should see the following output:
```text
ols() formula = PregN ~ Type

              sum_sq    df         F    PR(>F) Signif
Type       72.411467   3.0  3.539263  0.030507      *
Residual  156.856000  23.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

## **Exercise 1: One-Way ANOVA**

In the cell below, write the code to perform a one-way ANOVA using the `Cushings` dataset. Test whether the secretion of the metabolite Tetrahydrocortisone (`TCort`) is dependent upon the tumor type (`Type`). The `ols()` function should be `ols() formula = TCort ~ Type`.

In [ ]:
# @title Exercise 1: One-Way ANOVA



If the code is correct, you should see the following output:
```text
               sum_sq    df         F    PR(>F) Signif
Type       893.521000   3.0  3.225739  0.041218      *
Residual  2123.645667  23.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

## _Two-Way ANOVA (no interaction)_

For two-way ANOVA, we use also use `ols()`, but the right side of the formula includes both factors. We separate the two factors by the `+` sign if we do not want to include their interaction. For the “rat genotype” example discussed in Sect. 9.4.1, we use the formula `Wt ∼ C(Mother) + C(Litter)`

## Example 2: Two-Way ANOVA (no interaction)

The code in the cell below performs a two-way ANOVA on the rat `genotype` dataset with no interaction. The formula for the `ols()` function is `ols() formula = Wt ~ C(Mother) + C(Litter)`.

In [ ]:
# @title Example 2: Two-Way ANOVA (no interaction)

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Define your variables
dependent_var = "Wt"
factor_1 = "Mother"
factor_2 = "Litter"
dataset = genotype_df
interaction_symbol = "+"

# 1. Define the R-style significance function
def get_sig_stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    elif p_value < 0.1:
        return "."
    else:
        return " "

# 2, Define the formula
formula = f"{dependent_var} ~ C({factor_1}) {interaction_symbol} C({factor_2})"
print(f"ols() formula = {formula}\n")

# 3. Run Two-Way ANOVA
model = ols(formula, data=dataset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# 4. Add the significance stars column based on 'PR(>F)'
anova_table['Signif'] = anova_table['PR(>F)'].apply(get_sig_stars)

# 4. Print the updated table
print(f"ANOVA Table (Type II tests)")
print(f"\nResponse: {dependent_var}")
print(anova_table)
print("\nSignif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")

If the code is correct, you should see the following output:
```text
ols() formula = Wt ~ C(Mother) + C(Litter)

ANOVA Table (Type II tests)

Response: Wt
                sum_sq    df         F    PR(>F) Signif
C(Mother)   775.080588   3.0  4.273178  0.008861     **
C(Litter)    63.632488   3.0  0.350819  0.788698       
Residual   3264.889012  54.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

## **Exercise 2: Two-Way ANOVA (no interaction)**

In the cell below, write the code to perform a two-way ANOVA to evaluate the relationship between the
vitamin C content and cultivars in the `cabbages` dataset, while controlling for the effect of planting dates. The formula for the `ols()` function should be `ols() formula = VitC ~ C(Date) + C(Cult)`.

The `cabbages` dataset comes from a study comparing ascorbic acid (one form of vitamin C) content between two different cultivars (`c39` and `c52`) of cabbage. The two different cultivars were planted on three different dates, denoted
as `2.5`, `2.2`, or `4.3`. The variable `Date` is a factor that specifies the planting date for each cabbage.

**Code Hints:**

Change the section `# Define your variables` to read as follows:
```Python
# Define your variables
dependent_var = "VitC"
factor_1 = "Date"
factor_2 = "Cult"
dataset = cabbages_df
interaction_symbol = "+"
```

In [ ]:
# @title Exercise 2: Two-Way ANOVA (no interaction)



If the code is correct, you should see the following output:
```text
ols() formula = VitC ~ C(Date) + C(Cult)

ANOVA Table (Type II tests)

Response: VitC
           sum_sq    df          F        PR(>F) Signif
C(Date)    909.30   2.0   9.660924  2.485865e-04    ***
C(Cult)   2496.15   1.0  53.041056  1.179108e-09    ***
Residual  2635.40  56.0        NaN           NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

## _Two-Way ANOVA with Interaction_

For two-way ANOVA _with_ interaction, we again use `ols()`, and right side of the formula again contains both factors. However, in this instance, we separate the two factors by an asterisk `*` sign to tell the `ols()` that we also want to test for an interaction between the two factors.

## Example 3: Two-Way ANOVA with Interaction

The code in the cell below performs a two-way ANOVA with interaction using the `genotype` dataset. The code is exactly the same as in Example 2, except the interaction variable, `interaction_symbol`, has been changed form a `+` to an asterisks `*`.

In [ ]:
# @title Example 3: Two-Way ANOVA With Interaction

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Define your variables
dependent_var = "Wt"
factor_1 = "Mother"
factor_2 = "Litter"
dataset = genotype_df
interaction_symbol = "*"

# 1. Define the R-style significance function
def get_sig_stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    elif p_value < 0.1:
        return "."
    else:
        return " "

# 2, Define the formula
formula = f"{dependent_var} ~ C({factor_1}) {interaction_symbol} C({factor_2})"
print(f"ols() formula = {formula}\n")

# 3. Run Two-Way ANOVA
model = ols(formula, data=dataset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# 4. Add the significance stars column based on 'PR(>F)'
anova_table['Signif'] = anova_table['PR(>F)'].apply(get_sig_stars)

# 4. Print the updated table
print(f"ANOVA Table (Type II tests)")
print(f"\nResponse: {dependent_var}")
print(anova_table)
print("\nSignif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")

If the code is correction, you should see the following output:
```text
ols() formula = Wt ~ C(Mother) * C(Litter)

ANOVA Table (Type II tests)

Response: Wt
                          sum_sq    df         F    PR(>F) Signif
C(Mother)             775.080588   3.0  4.763246  0.005736     **
C(Litter)              63.632488   3.0  0.391052  0.760004       
C(Mother):C(Litter)   824.072512   9.0  1.688108  0.120053       
Residual             2440.816500  45.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

## **Exercise 3: Two-Way ANOVA With Interaction**

Repeat your two-way ANOVA on the `cabbages` dataset that you performed in **Execise 2** but this time, include in your analysis of a possible interaction between `Date` and `Cult`. Your `ols()` formula should be `ols() formula = VitC ~ C(Date) * C(Cult)`


In [ ]:
# @title Exercise 3: Two-Way ANOVA With Interaction



If the code is correct, you should see the following output:
```text
ols() formula = VitC ~ C(Date) * C(Cult)

ANOVA Table (Type II tests)

Response: VitC
                  sum_sq    df          F        PR(>F) Signif
C(Date)           909.30   2.0   9.855526  2.245180e-04    ***
C(Cult)          2496.15   1.0  54.109470  1.088693e-09    ***
C(Date):C(Cult)   144.30   2.0   1.564008  2.186275e-01       
Residual         2491.10  54.0        NaN           NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

```

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()